# Multi-Seed Training for Arabic ITSM Classification

Addresses **reviewer Issue #2**: single-seed experiments insufficient.

Runs 4 key configurations × 3 seeds = 12 training runs on Kaggle T4 GPU.

**Estimated time**: ~2 GPU hours total (≈10 min per run × 12 runs)

**Configurations**:
1. MARBERTv2  L1-only (seeds: 42, 123, 456)
2. MARBERTv2  L1+L2+L3 (seeds: 42, 123, 456)
3. MARBERTv2  all-5-heads (seeds: 42, 123, 456)
4. AraBERTv2  L1+L2+L3 (seeds: 42, 123, 456)

Note: seed=42 runs replicate the original experiments to confirm reproducibility.

In [ ]:
# Clone the repository
!git clone https://github.com/bazokhan/arabic-itsm-classification.git
%cd arabic-itsm-classification

# Install dependencies (ignoring torch to avoid overriding the cloud's optimized CUDA build)
!pip install transformers datasets accelerate evaluate arabert pyarabic statsmodels mlflow tqdm pyyaml -q

In [ ]:
# Link the processed dataset
!rm -rf data/processed && mkdir -p data/processed
!cp -rs /kaggle/input/datasets/mohamedalbaz/processed-data/* data/processed/
!ls data/processed
# Expected: label_encoders.pkl  test.csv  train.csv  val.csv

In [ ]:
!nvidia-smi

In [ ]:
import os
from pathlib import Path

# Directory for per-seed metrics
METRICS_DIR = Path('results/metrics/multi_seed')
METRICS_DIR.mkdir(parents=True, exist_ok=True)

SEEDS = [42, 123, 456]

CONFIGS = [
    {
        'name': 'marbert_l1',
        'model': 'UBC-NLP/MARBERTv2',
        'flags': '--task l1',
        'epochs': 5,
        'output_prefix': 'marbert_l1',
    },
    {
        'name': 'marbert_l1l2l3',
        'model': 'UBC-NLP/MARBERTv2',
        'flags': '--tasks l1 l2 l3',
        'epochs': 10,
        'output_prefix': 'marbert_l1_l2_l3',
    },
    {
        'name': 'marbert_5heads',
        'model': 'UBC-NLP/MARBERTv2',
        'flags': '--multi-task',
        'epochs': 10,
        'output_prefix': 'marbert_multi_task',
    },
    {
        'name': 'arabert_l1l2l3',
        'model': 'aubmindlab/bert-base-arabertv02',
        'flags': '--tasks l1 l2 l3',
        'epochs': 10,
        'output_prefix': 'arabert_l1_l2_l3',
    },
]

print(f'Total runs: {len(CONFIGS) * len(SEEDS)}')
for cfg in CONFIGS:
    for seed in SEEDS:
        print(f'  {cfg["name"]} seed={seed}')

In [ ]:
import json, subprocess, sys, time

run_log = []

for cfg in CONFIGS:
    for seed in SEEDS:
        run_key = f"{cfg['name']}_seed{seed}"
        out_metrics_path = METRICS_DIR / f"{run_key}_metrics.json"

        if out_metrics_path.exists():
            print(f'[SKIP] {run_key} — already exists')
            continue

        model_out_dir = f"models/{cfg['output_prefix']}_seed{seed}"
        cmd = (
            f"python scripts/train.py "
            f"{cfg['flags']} "
            f"--model {cfg['model']} "
            f"--epochs {cfg['epochs']} "
            f"--seed {seed} "
            f"--output-dir {model_out_dir} "
            f"--data-dir data/processed"
        )

        print(f'\n[RUN] {run_key}')
        print(f'  CMD: {cmd}')
        t0 = time.time()
        ret = os.system(cmd)
        elapsed = time.time() - t0
        print(f'  Time: {elapsed/60:.1f} min | Return code: {ret}')

        if ret != 0:
            print(f'  ERROR: run failed')
            run_log.append({'run': run_key, 'status': 'failed', 'time_min': elapsed/60})
            continue

        # Collect test metrics from the checkpoint dir
        test_metrics_file = Path(model_out_dir) / 'test_metrics.json'
        if test_metrics_file.exists():
            with open(test_metrics_file) as f:
                test_metrics = json.load(f)
            result = {'config': cfg['name'], 'seed': seed, 'metrics': test_metrics}
            with open(out_metrics_path, 'w') as f:
                json.dump(result, f, indent=2)
            print(f'  Saved metrics: {out_metrics_path}')
            run_log.append({'run': run_key, 'status': 'ok', 'time_min': elapsed/60})
        else:
            print(f'  WARNING: test_metrics.json not found at {test_metrics_file}')
            run_log.append({'run': run_key, 'status': 'no_metrics', 'time_min': elapsed/60})

print('\n=== Run log ===')
for r in run_log:
    print(f"  {r['run']}: {r['status']} ({r['time_min']:.1f} min)")

In [ ]:
# Aggregate results
!python scripts/aggregate_multi_seed.py --metrics-dir results/metrics/multi_seed

In [ ]:
# View summary
import json
with open('results/metrics/multi_seed_summary.json') as f:
    summary = json.load(f)

for config, data in summary['summary'].items():
    print(f'\n{config} (seeds: {data["seeds"]}):')
    for task, stats in data['tasks'].items():
        print(f'  {task}: {stats["mean"]*100:.2f} ± {stats["std"]*100:.2f} (F1%)')

In [ ]:
# Package all results for download
!zip -r multi_seed_results.zip results/metrics/multi_seed/ results/metrics/multi_seed_summary.json
print('Created multi_seed_results.zip — download from Kaggle output')